In [1]:
from pathlib import Path
from typing import Any, Dict, List, Tuple

import pandas as pd

In [2]:
def read_str(data: bytes, start: int, length: int) -> str:
    """
    Convierte una sección de bytes ASCII en string.
    """
    return (
        data[start : start + length]
        .decode(
            "ascii",
            errors="ignore",
        )
        .strip()
    )

In [3]:
def read_bdf_fixed_header(file_path: Path) -> Dict[str, Any]:
    """
    Lee el header fijo del archivo BDF.
    """
    with open(file_path, "rb") as file:
        fixed_header: bytes = file.read(256)

    return {
        "header_bytes": int(read_str(fixed_header, 184, 8)),
        "num_records": int(read_str(fixed_header, 236, 8)),
        "record_duration": float(read_str(fixed_header, 244, 8)),
        "num_channels": int(read_str(fixed_header, 252, 4)),
    }

In [4]:
def read_bdf_channel_headers(file_path: Path) -> pd.DataFrame:
    """
    Lee los headers de todos los canales.
    """
    fixed_header_info: Dict[str, Any] = read_bdf_fixed_header(file_path)

    header_bytes: int = fixed_header_info["header_bytes"]

    num_channels: int = fixed_header_info["num_channels"]

    with open(file_path, "rb") as file:
        full_header: bytes = file.read(header_bytes)

    signal_header_start: int = 256

    field_sizes: List[Tuple[str, int]] = [
        ("label", 16),
        ("transducer", 80),
        ("physical_dimension", 8),
        ("physical_min", 8),
        ("physical_max", 8),
        ("digital_min", 8),
        ("digital_max", 8),
        ("prefiltering", 80),
        ("samples_per_record", 8),
        ("reserved", 32),
    ]

    field_starts: Dict[str, int] = {}

    current_start: int = signal_header_start

    for field_name, field_size in field_sizes:
        field_starts[field_name] = current_start
        current_start += field_size * num_channels

    rows: List[Dict[str, Any]] = []

    participant: str = file_path.stem

    for channel_index in range(num_channels):

        row: Dict[str, Any] = {
            "participant": participant,
            "channel_index": channel_index + 1,
        }

        for field_name, field_size in field_sizes:

            start: int = field_starts[field_name] + channel_index * field_size

            value: str = read_str(
                full_header,
                start,
                field_size,
            )

            row[field_name] = value

        rows.append(row)

    return pd.DataFrame(rows)

In [5]:
dataset_dir: Path = Path("../dataset")

bdf_files: List[Path] = sorted(dataset_dir.glob("s*.bdf"))

print(f"Archivos encontrados: {len(bdf_files)}")

Archivos encontrados: 32


In [6]:
all_headers: List[pd.DataFrame] = []

for file_path in bdf_files:

    print(f"Leyendo: {file_path.name}")

    participant_df: pd.DataFrame = read_bdf_channel_headers(file_path)

    all_headers.append(participant_df)

all_headers_df: pd.DataFrame = pd.concat(
    all_headers,
    ignore_index=True,
)

all_headers_df.head()

Leyendo: s01.bdf
Leyendo: s02.bdf
Leyendo: s03.bdf
Leyendo: s04.bdf
Leyendo: s05.bdf
Leyendo: s06.bdf
Leyendo: s07.bdf
Leyendo: s08.bdf
Leyendo: s09.bdf
Leyendo: s10.bdf
Leyendo: s11.bdf
Leyendo: s12.bdf
Leyendo: s13.bdf
Leyendo: s14.bdf
Leyendo: s15.bdf
Leyendo: s16.bdf
Leyendo: s17.bdf
Leyendo: s18.bdf
Leyendo: s19.bdf
Leyendo: s20.bdf
Leyendo: s21.bdf
Leyendo: s22.bdf
Leyendo: s23.bdf
Leyendo: s24.bdf
Leyendo: s25.bdf
Leyendo: s26.bdf
Leyendo: s27.bdf
Leyendo: s28.bdf
Leyendo: s29.bdf
Leyendo: s30.bdf
Leyendo: s31.bdf
Leyendo: s32.bdf


,participant,channel_index,label,transducer,physical_dimension,physical_min,physical_max,digital_min,digital_max,prefiltering,samples_per_record,reserved
0,s01,1,Fp1,Active Electrode,uV,-262144,262143,-8388608,8388607,HP: DC; LP: 104 Hz,512,MON
1,s01,2,AF3,Active Electrode,uV,-262144,262143,-8388608,8388607,HP: DC; LP: 104 Hz,512,MON
2,s01,3,F7,Active Electrode,uV,-262144,262143,-8388608,8388607,HP: DC; LP: 104 Hz,512,MON
3,s01,4,F3,Active Electrode,uV,-262144,262143,-8388608,8388607,HP: DC; LP: 104 Hz,512,MON
4,s01,5,FC1,Active Electrode,uV,-262144,262143,-8388608,8388607,HP: DC; LP: 104 Hz,512,MON


In [7]:
channels_count_df: pd.DataFrame = (
    all_headers_df.groupby("participant")
    .agg(num_channels=("channel_index", "count"))
    .reset_index()
)

channels_count_df

,participant,num_channels
0,s01,48
1,s02,48
2,s03,48
3,s04,48
4,s05,48
5,s06,48
6,s07,48
7,s08,48
8,s09,48
9,s10,48


In [8]:
participant_channels: Dict[str, List[str]] = {}

for participant in sorted(all_headers_df["participant"].unique()):

    labels: List[str] = (
        all_headers_df[all_headers_df["participant"] == participant]
        .sort_values("channel_index")["label"]
        .tolist()
    )

    participant_channels[participant] = labels

In [9]:
reference_labels_48: List[str] = participant_channels["s01"]

reference_labels_49: List[str] = participant_channels["s29"]

In [10]:
group_48_consistency: List[Tuple[str, bool]] = []

for participant in sorted(participant_channels.keys()):

    participant_number: int = int(participant.replace("s", ""))

    if participant_number <= 28:

        is_equal: bool = participant_channels[participant] == reference_labels_48

        group_48_consistency.append((participant, is_equal))

group_48_df: pd.DataFrame = pd.DataFrame(
    group_48_consistency,
    columns=["participant", "same_channels_as_s01"],
)

group_48_df

,participant,same_channels_as_s01
0,s01,True
1,s02,True
2,s03,True
3,s04,True
4,s05,True
5,s06,True
6,s07,True
7,s08,True
8,s09,True
9,s10,True


In [11]:
group_49_consistency: List[Tuple[str, bool]] = []

for participant in sorted(participant_channels.keys()):

    participant_number: int = int(participant.replace("s", ""))

    if participant_number >= 29:

        is_equal: bool = participant_channels[participant] == reference_labels_49

        group_49_consistency.append((participant, is_equal))

group_49_df: pd.DataFrame = pd.DataFrame(
    group_49_consistency,
    columns=["participant", "same_channels_as_s29"],
)

group_49_df

,participant,same_channels_as_s29
0,s29,True
1,s30,True
2,s31,True
3,s32,True


In [12]:
channels_s01: List[str] = participant_channels["s01"]

channels_s29: List[str] = participant_channels["s29"]

only_in_s29: List[str] = [
    channel for channel in channels_s29 if channel not in channels_s01
]

only_in_s01: List[str] = [
    channel for channel in channels_s01 if channel not in channels_s29
]

print("Canales solo en s29:")
print(only_in_s29)

print("\nCanales solo en s01:")
print(only_in_s01)

Canales solo en s29:
['', '']

Canales solo en s01:
['Status']


In [13]:
all_headers_df[all_headers_df["participant"] == "s29"][
    [
        "channel_index",
        "label",
        "physical_dimension",
        "samples_per_record",
    ]
]

,channel_index,label,physical_dimension,samples_per_record
1344,1,Fp1,uV,512
1345,2,AF3,uV,512
1346,3,F3,uV,512
1347,4,F7,uV,512
1348,5,FC5,uV,512
1349,6,FC1,uV,512
1350,7,C3,uV,512
1351,8,T7,uV,512
1352,9,CP5,uV,512
1353,10,CP1,uV,512


In [14]:
print("Canales solo en s29:")
print(only_in_s29)

print("\nCanales solo en s01:")
print(only_in_s01)

Canales solo en s29:
['', '']

Canales solo en s01:
['Status']


In [15]:
all_headers_df[all_headers_df["participant"] == "s29"][
    [
        "channel_index",
        "label",
        "physical_dimension",
        "samples_per_record",
    ]
]

,channel_index,label,physical_dimension,samples_per_record
1344,1,Fp1,uV,512
1345,2,AF3,uV,512
1346,3,F3,uV,512
1347,4,F7,uV,512
1348,5,FC5,uV,512
1349,6,FC1,uV,512
1350,7,C3,uV,512
1351,8,T7,uV,512
1352,9,CP5,uV,512
1353,10,CP1,uV,512


In [16]:
comparison_s01_s23_df: pd.DataFrame = pd.DataFrame(
    {
        "channel_index": range(
            1,
            max(len(participant_channels["s01"]), len(participant_channels["s23"])) + 1,
        ),
        "s01_label": participant_channels["s01"]
        + [""]
        * (
            max(len(participant_channels["s01"]), len(participant_channels["s23"]))
            - len(participant_channels["s01"])
        ),
        "s23_label": participant_channels["s23"]
        + [""]
        * (
            max(len(participant_channels["s01"]), len(participant_channels["s23"]))
            - len(participant_channels["s23"])
        ),
    }
)

comparison_s01_s23_df["same_label"] = (
    comparison_s01_s23_df["s01_label"] == comparison_s01_s23_df["s23_label"]
)

comparison_s01_s23_df[comparison_s01_s23_df["same_label"] == False]

,channel_index,s01_label,s23_label,same_label
2,3,F7,F3,False
3,4,F3,F7,False
4,5,FC1,FC5,False
5,6,FC5,FC1,False
6,7,T7,C3,False
7,8,C3,T7,False
8,9,CP1,CP5,False
9,10,CP5,CP1,False
10,11,P7,P3,False
11,12,P3,P7,False


In [17]:
all_headers_df[all_headers_df["participant"] == "s23"][
    [
        "channel_index",
        "label",
        "physical_dimension",
        "samples_per_record",
    ]
]

,channel_index,label,physical_dimension,samples_per_record
1056,1,Fp1,uV,512
1057,2,AF3,uV,512
1058,3,F3,uV,512
1059,4,F7,uV,512
1060,5,FC5,uV,512
1061,6,FC1,uV,512
1062,7,C3,uV,512
1063,8,T7,uV,512
1064,9,CP5,uV,512
1065,10,CP1,uV,512


In [27]:
from typing import Any, Dict, List

import pandas as pd

In [28]:
def sorted_unique(values: pd.Series) -> List[str]:
    """
    Retorna valores únicos ordenados como lista de strings.
    """
    return sorted(
        list({str(value).strip() for value in values if str(value).strip() != ""})
    )

In [29]:
consolidated_channels_df: pd.DataFrame = (
    all_headers_df.groupby("label")
    .agg(
        units=("physical_dimension", sorted_unique),
        digital_min=("digital_min", sorted_unique),
        digital_max=("digital_max", sorted_unique),
        physical_min=("physical_min", sorted_unique),
        physical_max=("physical_max", sorted_unique),
        samples_per_record=("samples_per_record", sorted_unique),
        participants=("participant", sorted_unique),
    )
    .reset_index()
)

consolidated_channels_df

,label,units,digital_min,digital_max,physical_min,physical_max,samples_per_record,participants
0,,[Boolean],[-8388608],[8388607],[-8388608],[8388607],[512],"[s24, s25, s26, s27, s28, s29, s30, s31, s32]"
1,AF3,[uV],[-8388608],[8388607],[-262144],[262143],[512],"[s01, s02, s03, s04, s05, s06, s07, s08, s09, ..."
2,AF4,[uV],[-8388608],[8388607],[-262144],[262143],[512],"[s01, s02, s03, s04, s05, s06, s07, s08, s09, ..."
3,C3,[uV],[-8388608],[8388607],[-262144],[262143],[512],"[s01, s02, s03, s04, s05, s06, s07, s08, s09, ..."
4,C4,[uV],[-8388608],[8388607],[-262144],[262143],[512],"[s01, s02, s03, s04, s05, s06, s07, s08, s09, ..."
5,CP1,[uV],[-8388608],[8388607],[-262144],[262143],[512],"[s01, s02, s03, s04, s05, s06, s07, s08, s09, ..."
6,CP2,[uV],[-8388608],[8388607],[-262144],[262143],[512],"[s01, s02, s03, s04, s05, s06, s07, s08, s09, ..."
7,CP5,[uV],[-8388608],[8388607],[-262144],[262143],[512],"[s01, s02, s03, s04, s05, s06, s07, s08, s09, ..."
8,CP6,[uV],[-8388608],[8388607],[-262144],[262143],[512],"[s01, s02, s03, s04, s05, s06, s07, s08, s09, ..."
9,Cz,[uV],[-8388608],[8388607],[-262144],[262143],[512],"[s01, s02, s03, s04, s05, s06, s07, s08, s09, ..."


In [30]:
def participants_to_range(participants: List[str]) -> str:
    """
    Convierte lista de participantes
    en representación compacta.
    """

    if participants == [f"s{i:02d}" for i in range(1, 33)]:
        return "s01-s32"

    if participants == [f"s{i:02d}" for i in range(1, 29)]:
        return "s01-s28"

    if participants == [f"s{i:02d}" for i in range(24, 33)]:
        return "s24-s32"

    return ", ".join(participants)

In [31]:
consolidated_channels_df["Available in"] = consolidated_channels_df[
    "participants"
].apply(participants_to_range)

In [32]:
consolidated_channels_df["Digital type"] = "int24"

consolidated_channels_df["Physical type"] = "float32"

In [33]:
consolidated_channels_df = consolidated_channels_df[
    [
        "label",
        "units",
        "Digital type",
        "Physical type",
        "digital_min",
        "digital_max",
        "physical_min",
        "physical_max",
        "samples_per_record",
        "Available in",
    ]
]

In [34]:
consolidated_channels_df = consolidated_channels_df.rename(
    columns={
        "label": "Channel",
        "units": "Unit(s)",
        "digital_min": "Digital min",
        "digital_max": "Digital max",
        "physical_min": "Physical min",
        "physical_max": "Physical max",
        "samples_per_record": "Samples/record",
    }
)

In [35]:
consolidated_channels_df

,Channel,Unit(s),Digital type,Physical type,Digital min,Digital max,Physical min,Physical max,Samples/record,Available in
0,,[Boolean],int24,float32,[-8388608],[8388607],[-8388608],[8388607],[512],s24-s32
1,AF3,[uV],int24,float32,[-8388608],[8388607],[-262144],[262143],[512],s01-s32
2,AF4,[uV],int24,float32,[-8388608],[8388607],[-262144],[262143],[512],s01-s32
3,C3,[uV],int24,float32,[-8388608],[8388607],[-262144],[262143],[512],s01-s32
4,C4,[uV],int24,float32,[-8388608],[8388607],[-262144],[262143],[512],s01-s32
5,CP1,[uV],int24,float32,[-8388608],[8388607],[-262144],[262143],[512],s01-s32
6,CP2,[uV],int24,float32,[-8388608],[8388607],[-262144],[262143],[512],s01-s32
7,CP5,[uV],int24,float32,[-8388608],[8388607],[-262144],[262143],[512],s01-s32
8,CP6,[uV],int24,float32,[-8388608],[8388607],[-262144],[262143],[512],s01-s32
9,Cz,[uV],int24,float32,[-8388608],[8388607],[-262144],[262143],[512],s01-s32


In [36]:
print(consolidated_channels_df.to_string(index=False))

Channel            Unit(s) Digital type Physical type Digital min Digital max        Physical min       Physical max Samples/record                                                                                                      Available in
                 [Boolean]        int24       float32  [-8388608]   [8388607]          [-8388608]          [8388607]          [512]                                                                                                           s24-s32
    AF3               [uV]        int24       float32  [-8388608]   [8388607]           [-262144]           [262143]          [512]                                                                                                           s01-s32
    AF4               [uV]        int24       float32  [-8388608]   [8388607]           [-262144]           [262143]          [512]                                                                                                           s01-s32
     C3         

In [37]:
channel_order: List[str] = [
    # ============================================================
    # EEG (32)
    # ============================================================
    "Fp1",
    "AF3",
    "F7",
    "F3",
    "FC1",
    "FC5",
    "T7",
    "C3",
    "CP1",
    "CP5",
    "P7",
    "P3",
    "Pz",
    "PO3",
    "O1",
    "Oz",
    "O2",
    "PO4",
    "P4",
    "P8",
    "CP6",
    "CP2",
    "C4",
    "T8",
    "FC6",
    "FC2",
    "F4",
    "F8",
    "AF4",
    "Fp2",
    "Fz",
    "Cz",
    # ============================================================
    # EXG (8)
    # ============================================================
    "EXG1",
    "EXG2",
    "EXG3",
    "EXG4",
    "EXG5",
    "EXG6",
    "EXG7",
    "EXG8",
    # ============================================================
    # Sensores fisiológicos importantes
    # ============================================================
    "Erg1",
    "Erg2",
    "GSR1",
    "GSR2",
    # ============================================================
    # Otros
    # ============================================================
    "Resp",
    "Plet",
    "Temp",
    "Status",
    "",
]

In [38]:
channel_order_map: Dict[str, int] = {
    channel: index for index, channel in enumerate(channel_order)
}

In [39]:
consolidated_channels_df["sort_order"] = consolidated_channels_df["Channel"].map(
    channel_order_map
)

In [40]:
consolidated_channels_df = (
    consolidated_channels_df.sort_values("sort_order")
    .drop(columns=["sort_order"])
    .reset_index(drop=True)
)

In [41]:
consolidated_channels_df

,Channel,Unit(s),Digital type,Physical type,Digital min,Digital max,Physical min,Physical max,Samples/record,Available in
0,Fp1,[uV],int24,float32,[-8388608],[8388607],[-262144],[262143],[512],s01-s32
1,AF3,[uV],int24,float32,[-8388608],[8388607],[-262144],[262143],[512],s01-s32
2,F7,[uV],int24,float32,[-8388608],[8388607],[-262144],[262143],[512],s01-s32
3,F3,[uV],int24,float32,[-8388608],[8388607],[-262144],[262143],[512],s01-s32
4,FC1,[uV],int24,float32,[-8388608],[8388607],[-262144],[262143],[512],s01-s32
5,FC5,[uV],int24,float32,[-8388608],[8388607],[-262144],[262143],[512],s01-s32
6,T7,[uV],int24,float32,[-8388608],[8388607],[-262144],[262143],[512],s01-s32
7,C3,[uV],int24,float32,[-8388608],[8388607],[-262144],[262143],[512],s01-s32
8,CP1,[uV],int24,float32,[-8388608],[8388607],[-262144],[262143],[512],s01-s32
9,CP5,[uV],int24,float32,[-8388608],[8388607],[-262144],[262143],[512],s01-s32


In [42]:
print(consolidated_channels_df.to_string(index=False))

Channel            Unit(s) Digital type Physical type Digital min Digital max        Physical min       Physical max Samples/record                                                                                                      Available in
    Fp1               [uV]        int24       float32  [-8388608]   [8388607]           [-262144]           [262143]          [512]                                                                                                           s01-s32
    AF3               [uV]        int24       float32  [-8388608]   [8388607]           [-262144]           [262143]          [512]                                                                                                           s01-s32
     F7               [uV]        int24       float32  [-8388608]   [8388607]           [-262144]           [262143]          [512]                                                                                                           s01-s32
     F3         

In [45]:
from pathlib import Path
import mne

# Ruta al archivo BDF
bdf_path: Path = Path("../dataset/s24.bdf")

# Cargar archivo
raw = mne.io.read_raw_bdf(bdf_path, preload=False, verbose=False)

# Mostrar nombres reales de canales
for idx, ch in enumerate(raw.ch_names):
    print(idx, repr(ch))

0 'Fp1'
1 'AF3'
2 'F3'
3 'F7'
4 'FC5'
5 'FC1'
6 'C3'
7 'T7'
8 'CP5'
9 'CP1'
10 'P3'
11 'P7'
12 'PO3'
13 'O1'
14 'Oz'
15 'Pz'
16 'Fp2'
17 'AF4'
18 'Fz'
19 'F4'
20 'F8'
21 'FC6'
22 'FC2'
23 'Cz'
24 'C4'
25 'T8'
26 'CP6'
27 'CP2'
28 'P4'
29 'P8'
30 'PO4'
31 'O2'
32 'EXG1'
33 'EXG2'
34 'EXG3'
35 'EXG4'
36 'EXG5'
37 'EXG6'
38 'EXG7'
39 'EXG8'
40 'GSR1'
41 'GSR2'
42 'Erg1'
43 'Erg2'
44 'Resp'
45 'Plet'
46 'Temp'
47 ''


/tmp/ipykernel_15220/4215804633.py:8: RuntimeWarning: Channels contain different highpass filters. Highest filter setting will be stored.
  raw = mne.io.read_raw_bdf(bdf_path, preload=False, verbose=False)
/tmp/ipykernel_15220/4215804633.py:8: RuntimeWarning: Channels contain different lowpass filters. Lowest filter setting will be stored.
  raw = mne.io.read_raw_bdf(bdf_path, preload=False, verbose=False)
